In [1]:
import sys
sys.path.append('/workspace/python')

import duckdb
import polars as pl
import json
import utils_functions as utils

In [2]:
#Connect to DuckDB
con = duckdb.connect('/workspace/data/pipeline.duckdb')
print('Connected to DuckDB!')

Connected to DuckDB!


In [3]:
last_batch = con.execute('SELECT MAX(batch_id) FROM raw.api_data').fetchone()[0]

**MODIFY THE LINE OF CODE BELOW FOR DEBUGGING PURPOSES, OTHERWISE LEAVE IT ALONE SO IT ALWAYS LOADS THE LAST BATCH**

In [4]:
historical_batch_to_load = last_batch

In [5]:
# Adding the bronze table into a polars dataframe for further processing
df = con.execute(f"""
                 SELECT id, batch_id, page, raw_json 
                 FROM raw.api_data
                 WHERE batch_id = {historical_batch_to_load}
                 """).pl()

In [6]:
#Taking only the data column
df = df.with_columns([
    pl.col('raw_json')
      .map_elements(
          lambda x: [json.dumps(anime) for anime in json.loads(x)['data']], 
          return_dtype=pl.List(pl.String)
      )
      .alias('anime_list')
])

In [7]:
#Splitting each json with 25 animes into 25 rows with 1 anime each
df = df.explode('anime_list')

In [8]:
df = df.drop('raw_json').rename({
    'anime_list': 'anime_json',
    'id' : 'raw_id'
    })

In [9]:
records = [json.loads(x) for x in df['anime_json'].to_list()]
df_flattened = pl.json_normalize(records)

In [10]:
# Removing unnecessary columns
df_flattened = df_flattened[[
    'mal_id', 'title', 'title_english', 'type', 'source', 'episodes', 'status', 'duration', 'score', 'scored_by', 'rank', 'popularity',
    'members', 'favorites', 'year', 'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
    ]]

In [11]:
# Renaming columns
df_flattened = df_flattened.rename({
    'mal_id': 'anime_id',
    'episodes': 'no_of_episodes'
})

In [12]:
df_flattened = df_flattened.with_columns([
    pl.col('producers')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('producer_id'),
              pl.element().struct.field('name').alias('producer_name')
          )
      )
      .alias('producers'),
    
    pl.col('licensors')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('licensor_id'),
              pl.element().struct.field('name').alias('licensor_name')
          )
      )
      .alias('licensors'),

    pl.col('studios')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('studio_id'),
              pl.element().struct.field('name').alias('studio_name')
          )
      )
      .alias('studios'),

    pl.col('genres')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('genre_id'),
              pl.element().struct.field('name').alias('genre_name')
          )
      )
      .alias('genres'),

    pl.col('themes')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('theme_id'),
              pl.element().struct.field('name').alias('theme_name')
          )
      )
      .alias('themes'),

    pl.col('demographics')
      .list.eval(
          pl.struct(
              pl.element().struct.field('mal_id').alias('demographic_id'),
              pl.element().struct.field('name').alias('demographic_name')
          )
      )
      .alias('demographics')
])


In [13]:
concatted_df = pl.concat([df.drop('anime_json'), df_flattened], how='horizontal')

In [14]:
# Extracting bridge tables for genres, themes, demographics, studios, producers and licensors
links_anime_genres       = utils.extract_bridge_table(concatted_df, 'genres',       'genre_id',       'genre_name')
links_anime_themes       = utils.extract_bridge_table(concatted_df, 'themes',       'theme_id',       'theme_name')
links_anime_demographics = utils.extract_bridge_table(concatted_df, 'demographics', 'demographic_id', 'demographic_name')
links_anime_studios      = utils.extract_bridge_table(concatted_df, 'studios',      'studio_id',      'studio_name')
links_anime_producers    = utils.extract_bridge_table(concatted_df, 'producers',    'producer_id',    'producer_name')
links_anime_licensors    = utils.extract_bridge_table(concatted_df, 'licensors',    'licensor_id',    'licensor_name')

In [15]:
# Dropping the list[struct] columns as we already have all that information in the links tables
silver_anime = concatted_df.drop([
    'producers', 'licensors', 'studios', 'genres', 'themes', 'demographics'
])

In [16]:
#Dropping duplicates:

links_anime_genres = links_anime_genres.unique(subset=['anime_id', 'genre_id'], keep='last')
links_anime_themes = links_anime_themes.unique(subset=['anime_id', 'theme_id'], keep='last')
links_anime_demographics = links_anime_demographics.unique(subset=['anime_id', 'demographic_id'], keep='last')
links_anime_studios = links_anime_studios.unique(subset=['anime_id', 'studio_id'], keep='last')
links_anime_producers = links_anime_producers.unique(subset=['anime_id', 'producer_id'], keep='last')
links_anime_licensors = links_anime_licensors.unique(subset=['anime_id', 'licensor_id'], keep='last')

silver_anime = silver_anime.unique(subset='anime_id', keep='last')

**The cells below overwrite the merge tables in DuckDB with our dataframes.** 

**You can run them one by one for debugging purposes, but the notebook will run all of them when called.**

In [17]:
#import importlib
#importlib.reload(utils)

utils.overwrite_table(con, silver_anime, 'curated.anime_merge')

2026-03-22 18:09:37.213 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.anime_merge: 7203 rows written


In [18]:
utils.overwrite_table(con, links_anime_genres, 'curated.links_anime_genres_merge')

2026-03-22 18:09:37.315 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_genres_merge: 16462 rows written


In [19]:
utils.overwrite_table(con, links_anime_themes, 'curated.links_anime_themes_merge')

2026-03-22 18:09:37.368 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_themes_merge: 9389 rows written


In [20]:
utils.overwrite_table(con, links_anime_demographics, 'curated.links_anime_demographics_merge')

2026-03-22 18:09:37.404 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_demographics_merge: 2728 rows written


In [21]:
utils.overwrite_table(con, links_anime_studios, 'curated.links_anime_studios_merge')

2026-03-22 18:09:37.450 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_studios_merge: 7752 rows written


In [22]:
utils.overwrite_table(con, links_anime_producers, 'curated.links_anime_producers_merge')

2026-03-22 18:09:37.524 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_producers_merge: 21243 rows written


In [23]:
utils.overwrite_table(con, links_anime_licensors, 'curated.links_anime_licensors_merge')

2026-03-22 18:09:37.563 | INFO     | utils_functions:overwrite_table:53 - Overwrote curated.links_anime_licensors_merge: 4666 rows written


**The cells below upsert the merge tables into the destination curated/silver tables in DuckDB.** 

**You can run them one by one for debugging purposes, but the notebook will run all of them when called.**

In [24]:
utils.upsert_table(con, 'curated.anime_merge',                    'curated.anime',                    ['anime_id'])

2026-03-22 18:09:37.696 | INFO     | utils_functions:upsert_table:103 - Upserted curated.anime_merge → curated.anime: 7204 total rows now in table



        MERGE INTO curated.anime target
        USING curated.anime_merge src
        ON target.anime_id = src.anime_id
        WHEN MATCHED THEN
            UPDATE SET raw_id = src.raw_id, batch_id = src.batch_id, page = src.page, title = src.title, title_english = src.title_english, "type" = src."type", "source" = src."source", no_of_episodes = src.no_of_episodes, status = src.status, duration = src.duration, score = src.score, scored_by = src.scored_by, "rank" = src."rank", popularity = src.popularity, members = src.members, favorites = src.favorites, "year" = src."year", loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.raw_id, src.batch_id, src.page, src.anime_id, src.title, src.title_english, src."type", src."source", src.no_of_episodes, src.status, src.duration, src.score, src.scored_by, src."rank", src.popularity, src.members, src.favorites, src."year", src.loaded_at)
    


In [25]:
utils.upsert_table(con, 'curated.links_anime_genres_merge',       'curated.links_anime_genres',       ['anime_id', 'genre_id'])


        MERGE INTO curated.links_anime_genres target
        USING curated.links_anime_genres_merge src
        ON target.anime_id = src.anime_id AND target.genre_id = src.genre_id
        WHEN MATCHED THEN
            UPDATE SET genre_name = src.genre_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.genre_id, src.genre_name, src.loaded_at)
    


2026-03-22 18:09:38.037 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_genres_merge → curated.links_anime_genres: 16466 total rows now in table


In [26]:
utils.upsert_table(con, 'curated.links_anime_themes_merge',       'curated.links_anime_themes',       ['anime_id', 'theme_id'])

2026-03-22 18:09:38.221 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_themes_merge → curated.links_anime_themes: 9411 total rows now in table



        MERGE INTO curated.links_anime_themes target
        USING curated.links_anime_themes_merge src
        ON target.anime_id = src.anime_id AND target.theme_id = src.theme_id
        WHEN MATCHED THEN
            UPDATE SET theme_name = src.theme_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.theme_id, src.theme_name, src.loaded_at)
    


In [27]:
utils.upsert_table(con, 'curated.links_anime_demographics_merge', 'curated.links_anime_demographics', ['anime_id', 'demographic_id'])


        MERGE INTO curated.links_anime_demographics target
        USING curated.links_anime_demographics_merge src
        ON target.anime_id = src.anime_id AND target.demographic_id = src.demographic_id
        WHEN MATCHED THEN
            UPDATE SET demographic_name = src.demographic_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.demographic_id, src.demographic_name, src.loaded_at)
    


2026-03-22 18:09:38.265 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_demographics_merge → curated.links_anime_demographics: 2730 total rows now in table


In [28]:
utils.upsert_table(con, 'curated.links_anime_studios_merge',      'curated.links_anime_studios',      ['anime_id', 'studio_id'])

2026-03-22 18:09:38.426 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_studios_merge → curated.links_anime_studios: 7754 total rows now in table



        MERGE INTO curated.links_anime_studios target
        USING curated.links_anime_studios_merge src
        ON target.anime_id = src.anime_id AND target.studio_id = src.studio_id
        WHEN MATCHED THEN
            UPDATE SET studio_name = src.studio_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.studio_id, src.studio_name, src.loaded_at)
    


In [29]:
utils.upsert_table(con, 'curated.links_anime_producers_merge',    'curated.links_anime_producers',    ['anime_id', 'producer_id'])


        MERGE INTO curated.links_anime_producers target
        USING curated.links_anime_producers_merge src
        ON target.anime_id = src.anime_id AND target.producer_id = src.producer_id
        WHEN MATCHED THEN
            UPDATE SET producer_name = src.producer_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.producer_id, src.producer_name, src.loaded_at)
    


2026-03-22 18:09:39.011 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_producers_merge → curated.links_anime_producers: 21245 total rows now in table


In [30]:
utils.upsert_table(con, 'curated.links_anime_licensors_merge',    'curated.links_anime_licensors',    ['anime_id', 'licensor_id'])

2026-03-22 18:09:39.097 | INFO     | utils_functions:upsert_table:103 - Upserted curated.links_anime_licensors_merge → curated.links_anime_licensors: 4666 total rows now in table



        MERGE INTO curated.links_anime_licensors target
        USING curated.links_anime_licensors_merge src
        ON target.anime_id = src.anime_id AND target.licensor_id = src.licensor_id
        WHEN MATCHED THEN
            UPDATE SET licensor_name = src.licensor_name, loaded_at = src.loaded_at
        WHEN NOT MATCHED THEN
            INSERT VALUES (src.anime_id, src.licensor_id, src.licensor_name, src.loaded_at)
    


In [31]:
con.close()